# Weapon Detection — Custom Training Pipeline
## Overview
Trains a YOLOv11s model to detect **guns** and **knives** from two merged data sources:
- **Google Open Images v6** — large-scale, real-world images with bounding box annotations
- **Roboflow custom dataset** — additional weapon images with varied angles and conditions

## Final class mapping
| Class ID | Name   |
|----------|--------|
| 0        | gun    |
| 1        | knife  |

Set `WEAPON_ID_OFFSET = 0` in `app.py` (already the default).

## Pipeline order
1. Install deps & verify GPU
2. Create folder structure
3. Download Open Images metadata CSVs
4. Download Open Images train + val images (parallel, stream-filtered)
5. Download and merge Roboflow dataset
6. Write `data.yaml`
7. Train YOLOv11s for 50 epochs
8. Validate and export `weapon.pt`


In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CELL 1 — Reinstall PyTorch with CUDA support   ║
# ║                                                  ║
# ║  Kaggle's default PyTorch build sometimes ships  ║
# ║  without the correct CUDA toolkit version.       ║
# ║  We force-reinstall with cu118 to ensure GPU     ║
# ║  acceleration works correctly during training.   ║
# ╚══════════════════════════════════════════════════╝
!pip uninstall torch torchvision torchaudio -y -q
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
print('PyTorch reinstalled with CUDA 11.8 support.')


In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CELL 2 — Verify GPU availability               ║
# ║                                                  ║
# ║  Training will be ~50x slower on CPU.            ║
# ║  If no GPU is found, enable it in:               ║
# ║  Kaggle → Settings → Accelerator → GPU T4 x2    ║
# ╚══════════════════════════════════════════════════╝
import torch

print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())

if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
    print('VRAM    :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU found — enable GPU in Kaggle settings before training.')


In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CELL 3 — Install dependencies                   ║
# ║                                                  ║
# ║  ultralytics : YOLOv11 training framework        ║
# ║  roboflow    : used to download our custom       ║
# ║               weapon dataset via the API         ║
# ║  pyyaml      : read/write data.yaml config       ║
# ║  pandas      : process large annotation CSVs     ║
# ╚══════════════════════════════════════════════════╝
!pip install ultralytics roboflow pyyaml pandas -q
print('All dependencies installed.')


In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CELL 4 — Create dataset folder structure        ║
# ║                                                  ║
# ║  YOLO expects images and labels in matching      ║
# ║  subfolders for each split (train / val / test). ║
# ║  Structure:                                      ║
# ║    weapon_dataset/                               ║
# ║      train/images/   train/labels/               ║
# ║      val/images/     val/labels/                 ║
# ║      test/images/    test/labels/                ║
# ╚══════════════════════════════════════════════════╝
import os

ROOT = '/kaggle/working/weapon_dataset'

for split in ['train', 'val', 'test']:
    os.makedirs(f'{ROOT}/{split}/images', exist_ok=True)
    os.makedirs(f'{ROOT}/{split}/labels', exist_ok=True)

# Separate folder for Open Images metadata CSVs
os.makedirs('/kaggle/working/oi_meta', exist_ok=True)

print('Dataset folder structure created.')


In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CELL 5 — Download Open Images metadata CSVs    ║
# ║                                                  ║
# ║  We need 3 small CSV files from Google's         ║
# ║  Open Images v6 dataset (a few MB total):        ║
# ║                                                  ║
# ║  1. classes.csv   — maps class codes like        ║
# ║     '/m/0mkg' to human names like 'Handgun'      ║
# ║  2. val_bbox.csv  — bounding box annotations     ║
# ║     for the validation split (~35 MB)            ║
# ║  3. val_images.csv — image URLs for val split    ║
# ║                                                  ║
# ║  The train bbox CSV is ~800 MB so we stream-     ║
# ║  filter it later in Cell 7 instead of loading    ║
# ║  the whole file into memory at once.             ║
# ╚══════════════════════════════════════════════════╝
import urllib.request, os

META = '/kaggle/working/oi_meta'

downloads = [
    (
        'https://storage.googleapis.com/openimages/v5/class-descriptions-boxable.csv',
        f'{META}/classes.csv'
    ),
    (
        'https://storage.googleapis.com/openimages/v5/validation-annotations-bbox.csv',
        f'{META}/val_bbox.csv'
    ),
    (
        'https://storage.googleapis.com/openimages/2018_04/validation/validation-images-with-rotation.csv',
        f'{META}/val_images.csv'
    ),
]

for url, dst in downloads:
    if os.path.exists(dst):
        print(f'Already exists, skipping: {os.path.basename(dst)}')
        continue
    print(f'Downloading {os.path.basename(dst)} ...')
    urllib.request.urlretrieve(url, dst)
    size_mb = os.path.getsize(dst) / 1e6
    print(f'  Done — {size_mb:.1f} MB')

print('Metadata downloads complete.')


In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CELL 6 — (OPTIONAL) Reuse FiftyOne image cache ║
# ║                                                  ║
# ║  If you previously ran FiftyOne to explore Open  ║
# ║  Images, it may have already downloaded images   ║
# ║  to /root/fiftyone/. This cell copies those into ║
# ║  our dataset folder so nothing is re-downloaded. ║
# ║                                                  ║
# ║  If no FiftyOne cache exists, this cell does     ║
# ║  nothing — Cell 7 will download images fresh.   ║
# ╚══════════════════════════════════════════════════╝
import shutil, os, pandas as pd
from pathlib import Path

ROOT   = '/kaggle/working/weapon_dataset'
META   = '/kaggle/working/oi_meta'
OI_DIR = '/root/fiftyone/open-images-v7'

# Load class name -> OI class code mapping
# e.g. 'Handgun' -> '/m/0gxl3',  'Knife' -> '/m/04ctx'
classes_df   = pd.read_csv(f'{META}/classes.csv', header=None, names=['code', 'name'])
name_to_code = dict(zip(classes_df['name'], classes_df['code']))
HANDGUN_CODE = name_to_code.get('Handgun', '')
KNIFE_CODE   = name_to_code.get('Knife',   '')

print('Handgun code:', HANDGUN_CODE)
print('Knife code  :', KNIFE_CODE)

# Map OI class code -> our YOLO class id (0=gun, 1=knife)
OI_TO_YOLO = {HANDGUN_CODE: 0, KNIFE_CODE: 1}

def convert_oi_bbox_to_yolo(row, class_id):
    """
    Convert an Open Images bounding box row to YOLO format.
    Open Images stores coords as fractions (0.0–1.0) of image size,
    so no pixel conversion is needed — just compute center + size.
    Output format: 'class_id cx cy w h' (all normalized 0–1)
    """
    xmin, xmax = float(row['XMin']), float(row['XMax'])
    ymin, ymax = float(row['YMin']), float(row['YMax'])
    cx = max(0.0, min(1.0, (xmin + xmax) / 2.0))
    cy = max(0.0, min(1.0, (ymin + ymax) / 2.0))
    w  = max(0.001, min(1.0, xmax - xmin))
    h  = max(0.001, min(1.0, ymax - ymin))
    return f'{class_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}'

copied = skipped = 0

oi_img_dir = Path(OI_DIR) / 'train' / 'data'
if oi_img_dir.exists():
    oi_bbox_csv = Path(OI_DIR) / 'train' / 'labels' / 'detections.csv'
    if oi_bbox_csv.exists():
        print('Loading FiftyOne OI annotations...')
        bbox_df = pd.read_csv(str(oi_bbox_csv))
        bbox_df = bbox_df[bbox_df['LabelName'].isin(OI_TO_YOLO.keys())]
        print(f'Filtered annotations: {len(bbox_df)} rows')

        for img_id, group in bbox_df.groupby('ImageID'):
            img_candidates = list(oi_img_dir.glob(f'{img_id}.*'))
            if not img_candidates:
                skipped += 1
                continue
            img_path = img_candidates[0]
            lines = [
                convert_oi_bbox_to_yolo(row, OI_TO_YOLO[row['LabelName']])
                for _, row in group.iterrows()
                if row['LabelName'] in OI_TO_YOLO
            ]
            if not lines:
                skipped += 1
                continue
            shutil.copy(str(img_path), str(Path(ROOT) / 'train' / 'images' / img_path.name))
            dst_lbl = Path(ROOT) / 'train' / 'labels' / (img_path.stem + '.txt')
            with open(dst_lbl, 'w') as f:
                f.write('\n'.join(lines) + '\n')
            copied += 1

    print(f'Copied from FiftyOne cache: {copied} images, {skipped} skipped.')
else:
    print('No FiftyOne cache found — Cell 7 will download images from scratch.')


In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CELL 7 — Download Open Images train images     ║
# ║                                                  ║
# ║  Strategy:                                       ║
# ║  1. Load the train image manifest (~30 MB) to    ║
# ║     build an ImageID -> URL lookup table         ║
# ║  2. Stream the large train bbox CSV (~800 MB)    ║
# ║     in 50k-row chunks so we never load the whole ║
# ║     file into RAM — filter to Handgun/Knife only ║
# ║  3. Download up to MAX_ADDITIONAL images in      ║
# ║     parallel using 16 threads to save time       ║
# ╚══════════════════════════════════════════════════╝
import pandas as pd, urllib.request, os
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

ROOT = '/kaggle/working/weapon_dataset'
META = '/kaggle/working/oi_meta'

# Increase this to get more training images (each download takes ~1-2s)
MAX_ADDITIONAL = 3000

# ── Step 1: Load class codes ──────────────────────────────────────────────────
classes_df   = pd.read_csv(f'{META}/classes.csv', header=None, names=['code', 'name'])
name_to_code = dict(zip(classes_df['name'], classes_df['code']))
HANDGUN_CODE = name_to_code['Handgun']
KNIFE_CODE   = name_to_code['Knife']
OI_TO_YOLO   = {HANDGUN_CODE: 0, KNIFE_CODE: 1}

# ── Step 2: Load image URL manifest ──────────────────────────────────────────
# This maps each ImageID to its original download URL
print('Reading train image manifest...')
TRAIN_IMG_URL   = 'https://storage.googleapis.com/openimages/2018_04/train/train-images-boxable-with-rotation.csv'
TRAIN_IMG_LOCAL = f'{META}/train_images.csv'

if not os.path.exists(TRAIN_IMG_LOCAL):
    print('Downloading train image manifest (~30 MB)...')
    urllib.request.urlretrieve(TRAIN_IMG_URL, TRAIN_IMG_LOCAL)

img_manifest = pd.read_csv(TRAIN_IMG_LOCAL, usecols=['ImageID', 'OriginalURL'])
img_url_map  = dict(zip(img_manifest['ImageID'], img_manifest['OriginalURL']))
print(f'Loaded {len(img_url_map):,} image URLs.')

# ── Step 3: Stream-filter the large train bbox CSV ────────────────────────────
# The full CSV has ~1.7M rows for all classes — we only keep Handgun + Knife.
# Reading in 50k-row chunks avoids loading 800 MB into RAM at once.
TRAIN_BBOX_URL   = 'https://storage.googleapis.com/openimages/v6/oidv6-train-annotations-bbox.csv'
TRAIN_BBOX_LOCAL = f'{META}/train_bbox.csv'

if not os.path.exists(TRAIN_BBOX_LOCAL):
    print('Downloading & filtering train bbox annotations (~800 MB)...')
    target_codes  = {HANDGUN_CODE, KNIFE_CODE}
    filtered_rows = []
    total_read    = 0
    tmp_path      = f'{META}/train_bbox_raw.csv'

    urllib.request.urlretrieve(TRAIN_BBOX_URL, tmp_path)

    for chunk in pd.read_csv(tmp_path, chunksize=50000):
        filtered = chunk[chunk['LabelName'].isin(target_codes)]
        filtered_rows.append(filtered)
        total_read += len(chunk)
        if total_read % 500000 == 0:
            print(f'  Processed {total_read:,} rows...')

    bbox_filtered = pd.concat(filtered_rows, ignore_index=True)
    bbox_filtered.to_csv(TRAIN_BBOX_LOCAL, index=False)
    os.remove(tmp_path)
    print(f'Filtered to {len(bbox_filtered):,} weapon annotation rows.')
else:
    print('Loading cached filtered bbox CSV...')
    bbox_filtered = pd.read_csv(TRAIN_BBOX_LOCAL)
    print(f'Loaded {len(bbox_filtered):,} rows.')

# ── Step 4: Build per-image YOLO annotation dict ─────────────────────────────
# Skip images already copied from FiftyOne cache in Cell 6
already_have = {p.stem for p in Path(f'{ROOT}/train/images').glob('*.*')}
print(f'Already have {len(already_have)} train images from Cell 6.')

per_image = {}
for _, row in bbox_filtered.iterrows():
    img_id  = row['ImageID']
    yolo_id = OI_TO_YOLO.get(row['LabelName'])
    if yolo_id is None or img_id in already_have:
        continue
    if img_id not in per_image:
        per_image[img_id] = []
    xmin, xmax = float(row['XMin']), float(row['XMax'])
    ymin, ymax = float(row['YMin']), float(row['YMax'])
    cx = max(0.0, min(1.0, (xmin + xmax) / 2.0))
    cy = max(0.0, min(1.0, (ymin + ymax) / 2.0))
    w  = max(0.001, min(1.0, xmax - xmin))
    h  = max(0.001, min(1.0, ymax - ymin))
    per_image[img_id].append(f'{yolo_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')

img_ids_to_download = list(per_image.keys())[:MAX_ADDITIONAL]
print(f'Will download {len(img_ids_to_download)} additional images.')

# ── Step 5: Download images in parallel ───────────────────────────────────────
# ThreadPoolExecutor with 16 workers makes this ~16x faster than sequential.
# Each worker downloads one image + writes its label file.
def download_one(img_id):
    url = img_url_map.get(img_id)
    if not url:
        return False
    ext = url.split('.')[-1].split('?')[0].lower()
    if ext not in ('jpg', 'jpeg', 'png'):
        ext = 'jpg'
    dst_img = Path(ROOT) / 'train' / 'images' / f'{img_id}.{ext}'
    dst_lbl = Path(ROOT) / 'train' / 'labels' / f'{img_id}.txt'
    try:
        urllib.request.urlretrieve(url, str(dst_img))
        with open(dst_lbl, 'w') as f:
            f.write('\n'.join(per_image[img_id]) + '\n')
        return True
    except Exception:
        return False

ok = fail = 0
with ThreadPoolExecutor(max_workers=16) as executor:
    futures = {executor.submit(download_one, img_id): img_id for img_id in img_ids_to_download}
    for i, future in enumerate(as_completed(futures)):
        if future.result(): ok += 1
        else:               fail += 1
        if (i + 1) % 200 == 0:
            print(f'  {i+1}/{len(img_ids_to_download)} — ok:{ok}  fail:{fail}')

print(f'Download complete — {ok} succeeded, {fail} failed.')


In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CELL 8 — Download Open Images validation images ║
# ║                                                  ║
# ║  Same approach as Cell 7 but for the val split.  ║
# ║  Val bbox CSV is small (~35 MB) so we loaded it  ║
# ║  already in Cell 5 — no streaming needed here.   ║
# ╚══════════════════════════════════════════════════╝
import pandas as pd, urllib.request, os
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

ROOT = '/kaggle/working/weapon_dataset'
META = '/kaggle/working/oi_meta'

MAX_VAL = 800  # cap validation set size

# Load class codes
classes_df   = pd.read_csv(f'{META}/classes.csv', header=None, names=['code', 'name'])
name_to_code = dict(zip(classes_df['name'], classes_df['code']))
HANDGUN_CODE = name_to_code['Handgun']
KNIFE_CODE   = name_to_code['Knife']
OI_TO_YOLO   = {HANDGUN_CODE: 0, KNIFE_CODE: 1}

# Filter val annotations to Handgun + Knife only
val_bbox = pd.read_csv(f'{META}/val_bbox.csv')
val_bbox = val_bbox[val_bbox['LabelName'].isin(OI_TO_YOLO.keys())]
print(f'Val bbox rows after filtering: {len(val_bbox)}')

# Build ImageID -> URL map for val split
val_imgs    = pd.read_csv(f'{META}/val_images.csv', usecols=['ImageID', 'OriginalURL'])
val_url_map = dict(zip(val_imgs['ImageID'], val_imgs['OriginalURL']))

# Build per-image YOLO annotation dict
val_per_image = {}
for _, row in val_bbox.iterrows():
    img_id  = row['ImageID']
    yolo_id = OI_TO_YOLO.get(row['LabelName'])
    if yolo_id is None:
        continue
    if img_id not in val_per_image:
        val_per_image[img_id] = []
    xmin, xmax = float(row['XMin']), float(row['XMax'])
    ymin, ymax = float(row['YMin']), float(row['YMax'])
    cx = max(0.0, min(1.0, (xmin + xmax) / 2.0))
    cy = max(0.0, min(1.0, (ymin + ymax) / 2.0))
    w  = max(0.001, min(1.0, xmax - xmin))
    h  = max(0.001, min(1.0, ymax - ymin))
    val_per_image[img_id].append(f'{yolo_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')

val_ids = list(val_per_image.keys())[:MAX_VAL]
print(f'Downloading {len(val_ids)} val images...')

def download_val(img_id):
    url = val_url_map.get(img_id)
    if not url:
        return False
    ext = url.split('.')[-1].split('?')[0].lower()
    if ext not in ('jpg', 'jpeg', 'png'):
        ext = 'jpg'
    dst_img = Path(ROOT) / 'val' / 'images' / f'{img_id}.{ext}'
    dst_lbl = Path(ROOT) / 'val' / 'labels' / f'{img_id}.txt'
    try:
        urllib.request.urlretrieve(url, str(dst_img))
        with open(dst_lbl, 'w') as f:
            f.write('\n'.join(val_per_image[img_id]) + '\n')
        return True
    except Exception:
        return False

ok = fail = 0
with ThreadPoolExecutor(max_workers=16) as executor:
    futures = {executor.submit(download_val, img_id): img_id for img_id in val_ids}
    for i, future in enumerate(as_completed(futures)):
        if future.result(): ok += 1
        else:               fail += 1

print(f'Val download complete — {ok} succeeded, {fail} failed.')


In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CELL 9 — Dataset summary & class balance check  ║
# ║                                                  ║
# ║  Counts images and per-class annotations across  ║
# ║  all splits. Run this before training to confirm ║
# ║  the dataset looks healthy:                      ║
# ║  - train should have 500+ images ideally         ║
# ║  - gun vs knife counts should not be extreme     ║
# ║    (e.g. 10x imbalance may hurt the rarer class) ║
# ╚══════════════════════════════════════════════════╝
from pathlib import Path
from collections import Counter

ROOT        = '/kaggle/working/weapon_dataset'
CLASS_NAMES = {0: 'gun', 1: 'knife'}

total_train = 0
for split in ['train', 'val', 'test']:
    lbl_dir   = Path(ROOT) / split / 'labels'
    img_dir   = Path(ROOT) / split / 'images'
    lbl_files = list(lbl_dir.glob('*.txt'))
    img_files = list(img_dir.glob('*.*'))
    counts    = Counter()

    for lf in lbl_files:
        with open(lf) as f:
            for line in f:
                line = line.strip()
                if line:
                    counts[int(line.split()[0])] += 1

    print(f'[{split}]')
    print(f'  Images      : {len(img_files)}')
    print(f'  Label files : {len(lbl_files)}')
    for cid, name in CLASS_NAMES.items():
        print(f'  {name:8s} (class {cid}): {counts[cid]:,} annotations')
    print()

    if split == 'train':
        total_train = len(img_files)

print(f'Total train images: {total_train:,}')
if total_train < 500:
    print('WARNING: Low image count. Increase MAX_ADDITIONAL in Cell 7.')
elif total_train > 2000:
    print('Good dataset size — expect solid accuracy.')


In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CELL 10 — Download & merge Roboflow dataset    ║
# ║                                                  ║
# ║  Downloads a custom weapon detection dataset     ║
# ║  from Roboflow and merges it into our existing   ║
# ║  Open Images data.                               ║
# ║                                                  ║
# ║  The Roboflow dataset has 5 classes:             ║
# ║    grenade, knife, missile, pistol, rifle        ║
# ║  We remap these to our 2-class schema:           ║
# ║    pistol + rifle -> 0 (gun)                     ║
# ║    knife          -> 1 (knife)                   ║
# ║    grenade + missile -> DROPPED (out of scope)   ║
# ║                                                  ║
# ║  Set your API key in the environment variable    ║
# ║  ROBOFLOW_API_KEY before running this cell.      ║
# ║  In Kaggle: Add it under Add-ons > Secrets.     ║
# ╚══════════════════════════════════════════════════╝
import shutil, yaml, os
from pathlib import Path
from roboflow import Roboflow

ROOT = '/kaggle/working/weapon_dataset'

# Load API key from environment — never hardcode credentials in notebooks
api_key = os.environ.get('ROBOFLOW_API_KEY')
if not api_key:
    raise EnvironmentError(
        'ROBOFLOW_API_KEY not set. '
        'Add it under Kaggle > Add-ons > Secrets, then restart the session.'
    )

rf      = Roboflow(api_key=api_key)
project = rf.workspace('test-7awfy').project('weapon-detection-f1lih')
version = project.version(1)
version.download('yolov8', location='/kaggle/working/roboflow_dataset')

# Read the class names from Roboflow's data.yaml
RF_YAML = '/kaggle/working/roboflow_dataset/data.yaml'
with open(RF_YAML) as f:
    rf_cfg = yaml.safe_load(f)

rf_names = [n.lower().strip() for n in rf_cfg.get('names', [])]
print('Roboflow classes:', rf_names)

# Keywords used to remap Roboflow classes to our gun/knife schema
GUN_KEYWORDS   = {'gun', 'pistol', 'handgun', 'firearm', 'rifle', 'weapon', 'revolver'}
KNIFE_KEYWORDS = {'knife', 'blade', 'dagger', 'sword', 'machete', 'cleaver'}

# Build remap dict: Roboflow class id -> our YOLO id (or None to drop)
remap = {}
for old_id, name in enumerate(rf_names):
    if any(k in name for k in GUN_KEYWORDS):
        remap[old_id] = 0
    elif any(k in name for k in KNIFE_KEYWORDS):
        remap[old_id] = 1
    else:
        remap[old_id] = None  # class not relevant — will be dropped

print('Class remap:')
for old_id, new_id in remap.items():
    status = f'-> class {new_id}' if new_id is not None else '-> DROPPED'
    print(f'  {old_id} ({rf_names[old_id]}) {status}')

def remap_label_file(src_path, dst_path, remap):
    """
    Read a YOLO label file, remap class ids, drop unwanted classes,
    and write the result to dst_path.
    Returns True if at least one valid annotation was written.
    """
    lines_out = []
    with open(src_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts  = line.split()
            old_id = int(parts[0])
            new_id = remap.get(old_id)
            if new_id is None:
                continue  # drop this class
            parts[0] = str(new_id)
            lines_out.append(' '.join(parts))
    if lines_out:
        with open(dst_path, 'w') as f:
            f.write('\n'.join(lines_out) + '\n')
        return True
    return False

# Copy remapped images + labels into our merged dataset folder
split_map = {'train': 'train', 'valid': 'val', 'test': 'test'}
copied    = {'train': 0, 'val': 0, 'test': 0}

for rf_split, our_split in split_map.items():
    img_dir = Path('/kaggle/working/roboflow_dataset') / rf_split / 'images'
    lbl_dir = Path('/kaggle/working/roboflow_dataset') / rf_split / 'labels'
    if not img_dir.exists():
        continue
    for img_path in img_dir.glob('*.*'):
        lbl_path = lbl_dir / (img_path.stem + '.txt')
        if not lbl_path.exists():
            continue
        dst_lbl = Path(ROOT) / our_split / 'labels' / lbl_path.name
        ok = remap_label_file(str(lbl_path), str(dst_lbl), remap)
        if ok:
            shutil.copy(str(img_path), str(Path(ROOT) / our_split / 'images' / img_path.name))
            copied[our_split] += 1

print('Roboflow images merged:')
for split, n in copied.items():
    print(f'  {split}: {n} images')


In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CELL 11 — Write data.yaml                       ║
# ║                                                  ║
# ║  YOLO needs a YAML config file that tells it     ║
# ║  where the dataset lives and what the class      ║
# ║  names are. This must be written after all       ║
# ║  data has been downloaded and merged.            ║
# ╚══════════════════════════════════════════════════╝
import yaml

ROOT      = '/kaggle/working/weapon_dataset'
DATA_YAML = f'{ROOT}/data.yaml'

cfg = {
    'path'  : ROOT,
    'train' : 'train/images',
    'val'   : 'val/images',
    'test'  : 'test/images',
    'nc'    : 2,             # number of classes
    'names' : ['gun', 'knife'],
}

with open(DATA_YAML, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('data.yaml written:')
with open(DATA_YAML) as f:
    print(f.read())


In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CELL 12 — Train YOLOv11s                        ║
# ║                                                  ║
# ║  Model choice:                                   ║
# ║    yolo11s — fast + accurate, fits in <12GB VRAM ║
# ║    yolo11m — ~3-5% better mAP, needs 12GB+       ║
# ║                                                  ║
# ║  Key hyperparameters:                            ║
# ║    epochs=50     — enough for convergence on     ║
# ║                    this dataset size             ║
# ║    patience=15   — early stops if no improvement ║
# ║                    for 15 consecutive epochs     ║
# ║    batch=32      — reduce to 16 if OOM error     ║
# ║    imgsz=640     — standard YOLO input size      ║
# ║    optimizer=AdamW — better generalization than  ║
# ║                      default SGD for small data  ║
# ║    lr0=0.001     — initial learning rate         ║
# ║    lrf=0.01      — final LR = lr0 * lrf          ║
# ║    warmup_epochs=3 — ramps LR up slowly at start ║
# ║                      to avoid unstable early     ║
# ║                      gradient updates            ║
# ║                                                  ║
# ║  Augmentation (applied on-the-fly during training)║
# ║    mosaic=1.0    — combines 4 images into one,   ║
# ║                    greatly improves small-object ║
# ║                    detection                     ║
# ║    mixup=0.15    — blends two images + labels    ║
# ║    copy_paste=0.1 — copies objects across images ║
# ║    erasing=0.4   — randomly erases patches to    ║
# ║                    prevent over-reliance on       ║
# ║                    specific visual regions        ║
# ╚══════════════════════════════════════════════════╝
from ultralytics import YOLO

DATA_YAML = '/kaggle/working/weapon_dataset/data.yaml'

model = YOLO('yolo11s.pt')  # start from COCO pretrained weights

results = model.train(
    data          = DATA_YAML,
    epochs        = 50,
    imgsz         = 640,
    batch         = 32,
    device        = 0,           # GPU 0
    name          = 'weapon_model',
    project       = '/kaggle/working',
    patience      = 15,          # early stopping
    optimizer     = 'AdamW',
    lr0           = 0.001,
    lrf           = 0.01,
    warmup_epochs = 3,
    weight_decay  = 0.0005,
    # Colour augmentation
    hsv_h         = 0.015,       # hue shift
    hsv_s         = 0.7,         # saturation shift
    hsv_v         = 0.4,         # brightness shift
    # Geometric augmentation
    degrees       = 5.0,         # random rotation ±5°
    translate     = 0.1,         # random translation ±10%
    scale         = 0.5,         # random scale ±50%
    perspective   = 0.0005,      # slight perspective warp
    fliplr        = 0.5,         # horizontal flip 50% of the time
    # Advanced augmentation
    mosaic        = 1.0,
    mixup         = 0.15,
    copy_paste    = 0.1,
    erasing       = 0.4,
    save_period   = 10,          # save checkpoint every 10 epochs
    plots         = True,        # save training curves and confusion matrix
)

print('Training complete!')


In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CELL 13 — Validate best checkpoint             ║
# ║                                                  ║
# ║  Runs the best saved weights against the val     ║
# ║  set and reports:                                ║
# ║    mAP50    — mean average precision at IoU 0.5  ║
# ║               (main metric for detection tasks)  ║
# ║    mAP50-95 — averaged over IoU 0.5:0.95         ║
# ║               (stricter, COCO standard)          ║
# ║    Precision — of all predicted boxes, how many  ║
# ║               were correct                       ║
# ║    Recall    — of all real objects, how many     ║
# ║               were detected                      ║
# ╚══════════════════════════════════════════════════╝
from ultralytics import YOLO

best  = '/kaggle/working/weapon_model/weights/best.pt'
model = YOLO(best)

val_results = model.val(
    data    = '/kaggle/working/weapon_dataset/data.yaml',
    imgsz   = 640,
    conf    = 0.25,   # minimum confidence threshold for a detection to count
    iou     = 0.45,   # IoU threshold for NMS during evaluation
    verbose = True,
)

print()
print('=' * 40)
print(f'  mAP50     : {val_results.box.map50:.4f}')
print(f'  mAP50-95  : {val_results.box.map:.4f}')
print(f'  Precision : {val_results.box.mp:.4f}')
print(f'  Recall    : {val_results.box.mr:.4f}')
print('=' * 40)
for i, name in enumerate(['gun', 'knife']):
    try:
        print(f'  {name}: AP50 = {val_results.box.ap50[i]:.4f}')
    except Exception:
        pass


In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CELL 14 — Export final weapon.pt               ║
# ║                                                  ║
# ║  Copies the best checkpoint to weapon.pt and     ║
# ║  verifies the class mapping is correct before    ║
# ║  you download it.                                ║
# ║                                                  ║
# ║  Expected output:                                ║
# ║    Model classes: {0: 'gun', 1: 'knife'}         ║
# ║    Class mapping CORRECT.                        ║
# ╚══════════════════════════════════════════════════╝
import shutil
from ultralytics import YOLO

best   = '/kaggle/working/weapon_model/weights/best.pt'
output = '/kaggle/working/weapon.pt'
shutil.copy(best, output)

model = YOLO(output)
print('Model classes:', model.names)

if model.names == {0: 'gun', 1: 'knife'}:
    print('Class mapping CORRECT.')
else:
    print('WARNING: Unexpected class mapping — check Cells 10 and 11.')

print()
print('Next steps:')
print('  1. Download weapon.pt from Kaggle output panel')
print('  2. Place it next to app.py in your project folder')
print('  3. WEAPON_ID_OFFSET = 0 in app.py  (already the default)')
print('  4. streamlit run app.py')


In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CELL 15 — (OPTIONAL) Test on your own images   ║
# ║                                                  ║
# ║  Upload your own photos of guns or knives to     ║
# ║  /kaggle/working/test_imgs/ then run this cell   ║
# ║  to see how the model performs on real-world     ║
# ║  images outside the training distribution.      ║
# ║                                                  ║
# ║  Annotated results are saved to:                 ║
# ║  /kaggle/working/test_results/                   ║
# ╚══════════════════════════════════════════════════╝
from ultralytics import YOLO
from pathlib import Path
import os

TEST_DIR = '/kaggle/working/test_imgs'
os.makedirs(TEST_DIR, exist_ok=True)

test_images = list(Path(TEST_DIR).glob('*.*'))

if not test_images:
    print('No test images found.')
    print('Upload photos to:', TEST_DIR)
else:
    model   = YOLO('/kaggle/working/weapon.pt')
    results = model(
        [str(p) for p in test_images],
        conf    = 0.25,
        imgsz   = 640,
        save    = True,
        project = '/kaggle/working',
        name    = 'test_results',
    )
    for r in results:
        name = Path(r.path).name
        if r.boxes is None or len(r.boxes) == 0:
            print(f'{name}: no detections')
        else:
            for box in r.boxes:
                cid  = int(box.cls[0])
                conf = float(box.conf[0])
                print(f'{name}: {model.names[cid]} ({conf:.2f})')
    print('Annotated results saved to /kaggle/working/test_results/')
